# Notebook 1: Build the features and ECoG target

This notebook follows the feature extraction setup in Shimizu et al. It also extracts Qwen representations from the aligned transcript.

The analysis uses five interpretable feature groups:

- mel-spectrogram
- GBFB
- speech presence
- phonetic features
- syntactic features

## 0. Setup


In [1]:
!test -d /kaggle/working/EfficientAT || git clone --depth 1 https://github.com/fschmid56/EfficientAT.git /kaggle/working/EfficientAT
!pip install torchaudio torchvision bitsandbytes --quiet

import spacy
if not spacy.util.is_package("en_core_web_lg"):
    spacy.cli.download("en_core_web_lg")

import nltk
for package in ["cmudict", "punkt", "punkt_tab"]:
    nltk.download(package, quiet=True)

Cloning into '/kaggle/working/EfficientAT'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 57 (delta 7), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 2.32 MiB | 30.90 MiB/s, done.
Resolving deltas: 100% (7/7), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 1. Shared data and time grid


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import soundfile as sf
import librosa
from scipy.signal import resample_poly

ROOT = Path("/kaggle/input/datasets/delhialli/podcast-ecog-dataset/kaggle/working/ds005574")
OUT = Path("/kaggle/working/features")
OUT.mkdir(exist_ok=True)

AUDIO_SR = 16000
GRID_HZ = 4.0
WINDOW = 0.5
SEGMENT_SECONDS = 360
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 10)]

transcript = pd.read_csv(ROOT / "stimuli/podcast_transcript.csv")

raw0 = mne.io.read_raw_fif(
    ROOT / "derivatives/ecogprep/sub-01/ieeg/sub-01_task-podcast_desc-highgamma_ieeg.fif",
    preload=False,
    verbose="ERROR",
)
T = int(round(raw0.n_times / raw0.info["sfreq"] * GRID_HZ))
t = np.arange(T) / GRID_HZ

wav, source_sr = sf.read(ROOT / "stimuli/podcast.wav")
if wav.ndim == 2:
    wav = wav.mean(axis=1)
audio = librosa.resample(wav, orig_sr=source_sr, target_sr=AUDIO_SR).astype(np.float32)
audio32 = librosa.resample(wav, orig_sr=source_sr, target_sr=32000).astype(np.float32)

print("time points:", T)
print("words:", len(transcript))
print("audio at 16 kHz:", audio.shape)


def window_sum(offsets, vectors, grid, window=WINDOW):
    """Sum vectors whose offset lies in (time - window, time]."""
    offsets = np.asarray(offsets, float)
    vectors = np.asarray(vectors, float)

    order = np.argsort(offsets)
    sorted_offsets = offsets[order]
    sorted_vectors = vectors[order]

    cumulative = np.vstack([
        np.zeros(sorted_vectors.shape[1]),
        np.cumsum(sorted_vectors, axis=0),
    ])
    left = np.searchsorted(sorted_offsets, np.asarray(grid) - window, side="right")
    right = np.searchsorted(sorted_offsets, np.asarray(grid), side="right")
    return cumulative[right] - cumulative[left]

time points: 7200
words: 5136
audio at 16 kHz: (28800000,)


## 2. Phonetic features


In [3]:
from nltk.corpus import cmudict
from nltk import word_tokenize

CMU = cmudict.dict()
ARPABET = [
    "AA", "AE", "AH", "AO", "AW", "AY", "B", "CH", "D", "DH", "EH", "ER", "EY",
    "F", "G", "HH", "IH", "IY", "JH", "K", "L", "M", "N", "NG", "OW", "OY",
    "P", "R", "S", "SH", "T", "TH", "UH", "UW", "V", "W", "Y", "Z", "ZH",
]
PHONE_TO_INDEX = {phone: i for i, phone in enumerate(ARPABET)}


def word_phones(word):
    phones = []
    for token in word_tokenize(str(word)):
        pronunciation = CMU.get(token.lower())
        if pronunciation:
            phones.extend(phone.strip("012") for phone in pronunciation[0])
    return phones

phone_offsets = []
phone_vectors = []
oov_words = 0

for word, start, end in transcript[["word", "start", "end"]].itertuples(index=False):
    phones = word_phones(word)
    if not phones:
        oov_words += 1
        continue

    step = (end - start) / len(phones)
    for i, phone in enumerate(phones):
        phone_offsets.append(start + (i + 1) * step)
        vector = np.zeros(39, dtype=np.float32)
        vector[PHONE_TO_INDEX[phone]] = 1.0
        phone_vectors.append(vector)

phone_offsets = np.asarray(phone_offsets)
phone_vectors = np.asarray(phone_vectors)
phonetic = window_sum(phone_offsets, phone_vectors, t).astype(np.float32)

print("phonetic:", phonetic.shape)
print("words not found in CMUdict:", oov_words)

phonetic: (7200, 39)
words not found in CMUdict: 108


## 3. Syntactic features


In [4]:
import spacy
from sklearn.preprocessing import LabelBinarizer

nlp = spacy.load("en_core_web_lg")
tag_encoder = LabelBinarizer().fit(nlp.get_pipe("tagger").labels)
dep_encoder = LabelBinarizer().fit(nlp.get_pipe("parser").labels)

word_rows = transcript.copy()
word_rows["token"] = (word_rows["word"].astype(str) + " ").apply(nlp.tokenizer)
word_rows = word_rows.explode("token", ignore_index=True)

words = [token.text for token in word_rows["token"]]
spaces = [token.whitespace_ == " " for token in word_rows["token"]]
doc = nlp(spacy.tokens.Doc(nlp.vocab, words=words, spaces=spaces))

syntax_vectors = np.hstack([
    tag_encoder.transform([token.tag_ for token in doc]),
    dep_encoder.transform([token.dep_ for token in doc]),
    np.asarray([[int(token.is_stop)] for token in doc]),
]).astype(np.float32)

syntactic = window_sum(word_rows["end"].values, syntax_vectors, t).astype(np.float32)

print("syntactic:", syntactic.shape)
print(len(tag_encoder.classes_), "tag +", len(dep_encoder.classes_), "dependency + 1 stop-word feature")

syntactic: (7200, 96)
50 tag + 45 dependency + 1 stop-word feature


## 4. Timing controls

These three columns are used only as nuisance controls in the speech-versus-text comparison:

1. word count in the last 0.5 s
2. phoneme count in the last 0.5 s
3. time since the most recent word offset, clipped at 0.5 s


In [5]:
word_count = window_sum(
    transcript["end"].values,
    np.ones((len(transcript), 1), dtype=np.float32),
    t,
).astype(np.float32)

phoneme_count = phonetic.sum(axis=1, keepdims=True).astype(np.float32)

sorted_word_ends = np.sort(transcript["end"].values.astype(float))
last_word_index = np.searchsorted(sorted_word_ends, t, side="right") - 1
time_since_word = np.full(len(t), WINDOW, dtype=np.float32)
valid = last_word_index >= 0
time_since_word[valid] = np.minimum(
    t[valid] - sorted_word_ends[last_word_index[valid]],
    WINDOW,
)

timing = np.column_stack([
    word_count[:, 0],
    phoneme_count[:, 0],
    time_since_word,
]).astype(np.float32)

print("timing controls:", timing.shape)
print("columns: word_count, phoneme_count, time_since_word")

timing controls: (7200, 3)
columns: word_count, phoneme_count, time_since_word


## 5. Mel-spectrogram


In [6]:
HOP = 400                         # 25 ms at 16 kHz
mel = librosa.feature.melspectrogram(
    y=audio,
    sr=AUDIO_SR,
    hop_length=HOP,
    n_mels=32,
    fmin=0,
    fmax=8000,
)
mel_db = librosa.power_to_db(mel)

frames_per_grid_step = AUDIO_SR // HOP // int(GRID_HZ)
frames_per_window = int(WINDOW * AUDIO_SR / HOP) + 1
mel_padded = np.pad(
    mel_db,
    ((0, 0), (frames_per_window - 1, 0)),
    constant_values=mel_db.min(),
)
mel_feat = np.stack([
    mel_padded[:, i * frames_per_grid_step:i * frames_per_grid_step + frames_per_window].reshape(-1)
    for i in range(len(t))
]).astype(np.float32)

print("mel:", mel_feat.shape, "= 32 x", frames_per_window)

mel: (7200, 672) = 32 x 21


## 6. GBFB

This is a Python port of the Schädler et al. spectro-temporal Gabor filter bank. It produces 455 features at each time point.


In [7]:
from numpy.fft import fft2, ifft2


def _hz_to_mel(freq):
    return 2595 * np.log10(1 + freq / 700.0)


def _mel_to_hz(mel_value):
    return 700 * (10 ** (mel_value / 2595.0) - 1)


def _triangular_filterbank(fs, n_coeff, centres, width=(1, 1)):
    left_width, right_width = width
    indices = np.round(centres / fs * n_coeff).astype(int)
    n_bands = len(centres) - (left_width + right_width)
    matrix = np.zeros((n_bands, n_coeff))

    for i in range(n_bands):
        left = indices[i]
        centre = indices[i + left_width]
        right = indices[i + left_width + right_width]
        if left >= 1:
            matrix[i, left - 1:centre] = np.linspace(0, 1, centre - left + 1)
        if right <= n_coeff:
            matrix[i, centre - 1:right] = np.linspace(1, 0, right - centre + 1)
    return matrix


def log_mel_spectrogram(signal, fs, win_shift=10, win_length=25,
                        freq_range=None, num_bands=None, band_factor=1):
    signal = np.asarray(signal, float).ravel()
    if freq_range is None:
        freq_range = (64, min(fs // 2, 12000))
    if num_bands is None:
        centre_distance = (_hz_to_mel(4000) - _hz_to_mel(64)) / 24
        num_bands = int(np.floor(
            (_hz_to_mel(freq_range[1]) - _hz_to_mel(freq_range[0])) / centre_distance
        )) - 1
        freq_range = (
            freq_range[0],
            _mel_to_hz(_hz_to_mel(freq_range[0]) + centre_distance * (num_bands + 1)),
        )

    shift = round(win_shift / 1000 * fs)
    length = round(win_length / 1000 * fs)
    n_coeff = int(2 ** np.ceil(np.log2(length)))
    n_frames = 1 + (len(signal) - length) // shift
    frames = signal[np.arange(length)[:, None] + shift * np.arange(n_frames)[None, :]]

    window = np.hamming(length)
    window = window / np.sqrt(np.mean(window ** 2))
    spectrum = (1 / n_coeff) * np.abs(np.fft.fft(frames * window[:, None], n_coeff, axis=0))

    centres = _mel_to_hz(np.linspace(
        _hz_to_mel(freq_range[0]),
        _hz_to_mel(freq_range[1]),
        (num_bands + 1) * band_factor + 1,
    ))
    mel_spec = _triangular_filterbank(
        fs, n_coeff, centres, (band_factor, band_factor)
    ) @ spectrum
    return np.maximum(-20, np.minimum(0, 20 * np.log10(np.maximum(mel_spec, 0))) + 130)


def _hann_window(width):
    step = 1.0 / width

    def sequence(start, step_size, stop):
        count = int(np.floor((stop - start) / step_size + 1e-9)) + 1
        return start + step_size * np.arange(count)

    values = np.concatenate([
        sequence(0.5, -step, 0.0)[::-1],
        sequence(0.5, step, 1.0)[1:],
    ])
    values = values[(values > 0) & (values < 1)]
    return 0.5 * (1 - np.cos(2 * np.pi * values))


def _fft_convolve_2d(a, b):
    shape_a = np.array(a.shape)
    shape_b = np.array(b.shape)
    full_shape = shape_a + shape_b - 1
    fft_shape = (2 ** np.ceil(np.log2(full_shape))).astype(int)
    result = ifft2(fft2(a, fft_shape) * fft2(b, fft_shape))[:full_shape[0], :full_shape[1]]
    y_offset, x_offset = shape_b[0] // 2, shape_b[1] // 2
    return result[y_offset:y_offset + shape_a[0], x_offset:x_offset + shape_a[1]]


def _gabor_filter(omega_k, omega_n, nu_k, nu_n, max_k, max_n):
    width_n = np.inf if omega_n == 0 else 2 * np.pi / abs(omega_n) * nu_n / 2
    width_k = np.inf if omega_k == 0 else 2 * np.pi / abs(omega_k) * nu_k / 2
    if width_n > max_n:
        width_n, omega_n = max_n, 0
    if width_k > max_k:
        width_k, omega_k = max_k, 0

    envelope = np.outer(_hann_window(width_k), _hann_window(width_n))
    n_k, n_n = envelope.shape
    n, k = np.meshgrid(np.arange(1, n_n + 1), np.arange(1, n_k + 1))
    filt = envelope * np.exp(
        1j * omega_n * (n - (n_n + 1) / 2)
        + 1j * omega_k * (k - (n_k + 1) / 2)
    )
    if omega_n != 0 or omega_k != 0:
        filt = filt - envelope / envelope.mean() * filt.mean()
    else:
        filt = filt + 1j * filt
    return filt / np.max(np.abs(fft2(filt)))


def _apply_gabor(filt, log_mel):
    if np.any(filt.real < 0):
        weights = np.abs(filt) / np.abs(filt).sum()
        correction = (
            _fft_convolve_2d(log_mel, weights)
            / _fft_convolve_2d(np.ones_like(log_mel), weights)
            * _fft_convolve_2d(np.ones_like(log_mel), filt)
        )
    else:
        correction = 0
    return _fft_convolve_2d(log_mel, filt) - correction


def _reduce_gabor(filt, matrix):
    step = max(1, filt.shape[0] // 4)
    return matrix[(matrix.shape[0] // 2) % step::step, :]


def _gabor_axes(omega_max, size_max, nu, distance):
    omega_min = np.pi * np.array(nu) / np.array(size_max)
    constants = np.array(distance) * 8.0 / np.array(nu)

    def sequence(maximum, spacing, minimum):
        values = {1: maximum}
        count = 0
        while np.all(np.array([values[i] for i in sorted(values)]) / spacing > minimum):
            values[1 + count] = maximum / spacing ** count
            count += 1
        return np.array([values[i] for i in sorted(values)])

    omega_n = np.concatenate([
        [0],
        np.flip(sequence(
            omega_max[1],
            (1 + constants[1] / 2) / (1 - constants[1] / 2),
            omega_min[1],
        )),
    ])
    omega_k = sequence(
        omega_max[0],
        (1 + constants[0] / 2) / (1 - constants[0] / 2),
        omega_min[0],
    )
    omega_k = np.concatenate([-omega_k, [0], np.flip(omega_k)])
    return omega_n, omega_k


def gbfb(log_mel, omega_max=(np.pi / 2, np.pi / 2), size_max=None,
         nu=(3.5, 3.5), distance=(0.3, 0.2)):
    n_bands = log_mel.shape[0]
    if size_max is None:
        size_max = (3 * n_bands, 40)

    context = size_max[1] // 2
    padded = np.hstack([
        np.repeat(log_mel[:, :1], context, axis=1),
        log_mel,
        np.repeat(log_mel[:, -1:], context, axis=1),
    ])

    omega_n, omega_k = _gabor_axes(omega_max, size_max, nu, distance)
    outputs = []
    for value_k in omega_k:
        for value_n in omega_n:
            if not (value_k < 0 and value_n == 0):
                filt = _gabor_filter(
                    value_k, value_n, nu[0], nu[1], size_max[0], size_max[1]
                )
                outputs.append(np.real(_reduce_gabor(filt, _apply_gabor(filt, padded))))
    return np.vstack(outputs)[:, context:-context]

In [8]:
log_mel_gbfb = log_mel_spectrogram(audio, AUDIO_SR)
gabor = gbfb(log_mel_gbfb)
gbfb_feat = resample_poly(gabor, 1, 25, axis=1)[:, :len(t)].T.astype(np.float32)

print("GBFB:", gbfb_feat.shape)
print("all values finite:", np.isfinite(gbfb_feat).all())

GBFB: (7200, 455)
all values finite: True


## 7. Speech presence

EfficientAT `mn20_as` estimates the probability of the AudioSet speech class from the previous 0.5 s of audio.


In [9]:
import os
import sys
import torch

current_dir = os.getcwd()
os.chdir("/kaggle/working/EfficientAT")
sys.path.insert(0, "/kaggle/working/EfficientAT")
from models.mn.model import get_model as get_mobilenet
from models.preprocess import AugmentMelSTFT
from helpers.utils import NAME_TO_WIDTH
os.chdir(current_dir)

speech_model = get_mobilenet(
    width_mult=NAME_TO_WIDTH("mn20_as"),
    pretrained_name="mn20_as",
).eval()
speech_mel = AugmentMelSTFT(
    n_mels=128,
    sr=32000,
    win_length=800,
    hopsize=320,
).eval()

SPEECH_CLASS = 0
SPEECH_WINDOW = 0.5
clip_samples = int(SPEECH_WINDOW * 32000)
padded_audio32 = np.concatenate([np.zeros(clip_samples, np.float32), audio32])
end_samples32 = (t * 32000).astype(int)

speech_probability = np.empty(len(t), np.float32)
with torch.no_grad():
    for start in range(0, len(t), 128):
        rows = np.arange(start, min(start + 128, len(t)))
        clips = np.stack([
            padded_audio32[end:end + clip_samples]
            for end in end_samples32[rows]
        ])
        spectrogram = speech_mel(torch.from_numpy(clips))
        logits = speech_model(spectrogram.unsqueeze(1))[0]
        speech_probability[rows] = torch.sigmoid(logits)[:, SPEECH_CLASS].numpy()

speech_feat = speech_probability[:, None]
print("speech:", speech_feat.shape)
print("range:", round(float(speech_probability.min()), 3), "to", round(float(speech_probability.max()), 3))

/usr/local/lib/python3.12/dist-packages/torchvision/ops/misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(


Downloading: "https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/mn20_as_mAP_478.pt" to resources/mn20_as_mAP_478.pt


100%|██████████| 68.6M/68.6M [00:00<00:00, 448MB/s]


MN(
  (features): Sequential(
    (0): ConvNormActivation(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): ConvNormActivation(
          (0): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      

/usr/local/lib/python3.12/dist-packages/torch/functional.py:681: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:874.)
  return _VF.stft(  # type: ignore[attr-defined]
/kaggle/working/EfficientAT/models/preprocess.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


speech: (7200, 1)
range: 0.002 to 0.863


## 8. Whisper Large V1

We use a 30 s window and a 250 ms stride. The look-back is reset every 360 s so model context does not cross an outer cross-validation boundary.


In [10]:
from transformers import WhisperFeatureExtractor, WhisperModel

DEVICE = "cuda"
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-large")
whisper_encoder = (
    WhisperModel.from_pretrained("openai/whisper-large", torch_dtype=torch.float16)
    .to(DEVICE)
    .eval()
    .get_encoder()
)

whisper_window = 30 * AUDIO_SR
segment_samples = SEGMENT_SECONDS * AUDIO_SR
end_samples16 = (t * AUDIO_SR).astype(int)
n_whisper_layers = whisper_encoder.config.encoder_layers + 1
whisper_feat = np.empty(
    (len(t), n_whisper_layers, whisper_encoder.config.d_model),
    dtype=np.float16,
)


def whisper_window_ending_at(end_sample):
    segment_start = (end_sample // segment_samples) * segment_samples
    start_sample = max(end_sample - whisper_window, segment_start)
    clip = audio[start_sample:end_sample]
    return np.concatenate([
        np.zeros(whisper_window - len(clip), dtype=np.float32),
        clip,
    ])

batch_size = 8
with torch.no_grad():
    for start in range(0, len(t), batch_size):
        rows = np.arange(start, min(start + batch_size, len(t)))
        waves = [whisper_window_ending_at(end_samples16[row]) for row in rows]
        inputs = feature_extractor(
            waves,
            sampling_rate=AUDIO_SR,
            return_tensors="pt",
        ).input_features.to(DEVICE, torch.float16)

        hidden_states = whisper_encoder(
            inputs,
            output_hidden_states=True,
        ).hidden_states
        last_tokens = torch.stack([state[:, -1, :] for state in hidden_states], dim=1)
        whisper_feat[rows] = last_tokens.float().cpu().numpy().astype(np.float16)

        if start % 800 == 0:
            print(f"{start}/{len(t)}")

print("Whisper:", whisper_feat.shape)
print("all values finite:", np.isfinite(whisper_feat).all())

del whisper_encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

0/7200
800/7200
1600/7200
2400/7200
3200/7200
4000/7200
4800/7200
5600/7200
6400/7200
Whisper: (7200, 33, 1280)
all values finite: True


## 9. Qwen2.5-7B

Each transcript word is tokenized with Qwen. Subword tokens inherit that word's end time.

The model uses up to 256 earlier tokens. Context is reset every 360 s. Padding is masked before the forward pass.


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

QWEN_MODEL = "Qwen/Qwen2.5-7B"
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_words = transcript.copy()
qwen_words["token_id"] = qwen_words["word"].astype(str).apply(
    lambda word: qwen_tokenizer.encode(" " + word, add_special_tokens=False)
)
qwen_tokens = qwen_words.explode("token_id", ignore_index=True)
qwen_tokens["token_id"] = qwen_tokens["token_id"].astype(int)

print("Qwen tokens:", len(qwen_tokens))
print("mean tokens per word:", round(len(qwen_tokens) / len(qwen_words), 2))

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    quantization_config=quantization,
    device_map="cuda",
).eval()

print("layers:", qwen_model.config.num_hidden_layers + 1)
print("hidden size:", qwen_model.config.hidden_size)

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

layers: 29
hidden size: 3584


In [12]:
CONTEXT_TOKENS = 256
qwen_token_ids = qwen_tokens["token_id"].to_numpy(dtype=np.int64)
qwen_token_ends = qwen_tokens["end"].to_numpy(dtype=float)
qwen_segment = np.floor(qwen_token_ends / SEGMENT_SECONDS).astype(int)

segment_first_token = np.empty(len(qwen_token_ids), dtype=int)
for segment in np.unique(qwen_segment):
    positions = np.flatnonzero(qwen_segment == segment)
    segment_first_token[positions] = positions[0]

bos_id = qwen_tokenizer.bos_token_id
if bos_id is None:
    bos_id = qwen_tokenizer.eos_token_id

n_qwen_layers = qwen_model.config.num_hidden_layers + 1
qwen_hidden_size = qwen_model.config.hidden_size
qwen_token_embeddings = np.empty(
    (len(qwen_token_ids), n_qwen_layers, qwen_hidden_size),
    dtype=np.float16,
)

batch_size = 8
with torch.no_grad():
    for start in range(0, len(qwen_token_ids), batch_size):
        token_positions = list(range(start, min(start + batch_size, len(qwen_token_ids))))
        sequences = []

        for token_position in token_positions:
            left = max(
                segment_first_token[token_position],
                token_position - CONTEXT_TOKENS,
            )
            sequences.append([
                bos_id,
                *qwen_token_ids[left:token_position + 1].tolist(),
            ])

        max_length = max(len(sequence) for sequence in sequences)
        input_ids = np.full(
            (len(sequences), max_length),
            qwen_tokenizer.pad_token_id,
            dtype=np.int64,
        )
        attention_mask = np.zeros_like(input_ids)

        for row, sequence in enumerate(sequences):
            input_ids[row, -len(sequence):] = sequence
            attention_mask[row, -len(sequence):] = 1

        output = qwen_model(
            input_ids=torch.tensor(input_ids, device="cuda"),
            attention_mask=torch.tensor(attention_mask, device="cuda"),
            output_hidden_states=True,
            use_cache=False,
        )
        last_tokens = torch.stack(
            [state[:, -1, :] for state in output.hidden_states],
            dim=1,
        )
        qwen_token_embeddings[start:start + len(sequences)] = (
            last_tokens.float().cpu().numpy().astype(np.float16)
        )

        if start % 512 == 0:
            print(f"{start}/{len(qwen_token_ids)}")

order = np.argsort(qwen_token_ends, kind="stable")
sorted_ends = qwen_token_ends[order]
sorted_embeddings = qwen_token_embeddings[order]
sorted_segments = qwen_segment[order]

last_token_index = np.searchsorted(sorted_ends, t, side="right") - 1
safe_index = last_token_index.clip(0)
grid_segment = np.floor(t / SEGMENT_SECONDS).astype(int)

recent = last_token_index >= 0
recent &= (t - sorted_ends[safe_index]) <= WINDOW
recent &= sorted_segments[safe_index] == grid_segment

qwen_feat = np.zeros(
    (len(t), n_qwen_layers, qwen_hidden_size),
    dtype=np.float16,
)
qwen_feat[recent] = sorted_embeddings[safe_index[recent]]

np.save(OUT / "qwen.npy", qwen_feat)
print("Qwen:", qwen_feat.shape)
print("non-zero time points:", int(qwen_feat.any(axis=(1, 2)).sum()))
print("all values finite:", np.isfinite(qwen_feat).all())

del qwen_model, qwen_token_embeddings, sorted_embeddings
if torch.cuda.is_available():
    torch.cuda.empty_cache()

0/5501
512/5501
1024/5501
1536/5501
2048/5501
2560/5501
3072/5501
3584/5501
4096/5501
4608/5501
5120/5501
Qwen: (7200, 29, 3584)
non-zero time points: 5805
all values finite: True


## 10. Build the pooled ECoG target


In [13]:
def load_subject(subject):
    path = (
        ROOT / "derivatives/ecogprep" / subject / "ieeg"
        / f"{subject}_task-podcast_desc-highgamma_ieeg.fif"
    )
    raw = mne.io.read_raw_fif(path, preload=True, verbose="ERROR")
    downsample = round(raw.info["sfreq"] / GRID_HZ)
    signal = resample_poly(raw.get_data(), 1, downsample, axis=1).T
    signal = (signal - signal.mean(axis=0)) / signal.std(axis=0)
    return signal.astype(np.float32), raw.ch_names

subject_blocks = []
column_subject = []
column_channel = []

for subject in SUBJECTS:
    subject_signal, channel_names = load_subject(subject)
    subject_blocks.append(subject_signal)
    column_subject.extend([subject] * subject_signal.shape[1])
    column_channel.extend(channel_names)
    print(subject, ":", subject_signal.shape[1], "electrodes")

Y_all = np.concatenate(subject_blocks, axis=1)
print("pooled target:", Y_all.shape)

sub-01 : 99 electrodes
sub-02 : 90 electrodes
sub-03 : 235 electrodes
sub-04 : 143 electrodes
sub-05 : 159 electrodes
sub-06 : 166 electrodes
sub-07 : 116 electrodes
sub-08 : 72 electrodes
sub-09 : 188 electrodes
pooled target: (7200, 1268)


## 11. Save the arrays


In [ ]:
arrays = {
    "Y_all": Y_all,
    "col_subject": np.asarray(column_subject),
    "col_channel": np.asarray(column_channel),
    "t": t,
    "phonetic": phonetic,
    "syntactic": syntactic,
    "timing": timing,
    "mel": mel_feat,
    "gbfb": gbfb_feat,
    "speech": speech_feat,
    "whisper": whisper_feat,
}

for name, array in arrays.items():
    np.save(OUT / f"{name}.npy", array)

print("saved files:")
for path in sorted(OUT.glob("*.npy")):
    print(" ", path.name, f"({path.stat().st_size / 1e6:.1f} MB)")

saved files:
  Y_all.npy (36.5 MB)
  col_channel.npy (0.0 MB)
  col_subject.npy (0.0 MB)
  gbfb.npy (13.1 MB)
  mel.npy (19.4 MB)
  phonetic.npy (1.1 MB)
  qwen.npy (1496.7 MB)
  speech.npy (0.0 MB)
  syntactic.npy (2.8 MB)
  t.npy (0.1 MB)
  timing.npy (0.1 MB)
  whisper.npy (608.3 MB)


In [16]:
import shutil
from IPython.display import FileLink, display

zip_base = "/kaggle/working/features"
shutil.make_archive(zip_base, "zip", OUT)
print("archive size:", round(Path(zip_base + ".zip").stat().st_size / 1e6), "MB")
display(FileLink("features.zip"))

archive size: 1533 MB


/kaggle/working/features.zip